# slm-quant-bench: quality sweep on Colab (T4)

Reference-environment quality runs for the MSc dissertation. Efficiency
and feasibility are measured separately on the target device; this
notebook covers the quality half only.

**One model per session** keeps free-tier limits happy. Set `MODEL` and
`MODE` in the parameter cell, then `Runtime > Run all`. When it
finishes, `Runtime > Disconnect and delete runtime`, then start the
next model.

Run the **pilot first**: `MODE = "pilot"` with `MODEL = "phi3-mini"`.
That is the protocol gate, and its timings decide whether the free tier
is sufficient for the full sweep. Only once the protocol is frozen,
switch to `MODE = "full"` and work through the models in order:
`phi3-mini`, `gemma2-2b`, `llama32-3b`, `mistral7b`.

### One-time setup (first session only)

Store two secrets in Colab so no token is ever typed into a cell.
Click the **key icon** in the left sidebar and add:

| Name | Value | Notebook access |
|---|---|---|
| `HF_TOKEN` | a Hugging Face **read** token (huggingface.co/settings/tokens) | on |
| `GITHUB_TOKEN` | a GitHub fine-grained PAT with **Contents: read and write** on the dissertation repo | on |

Also confirm **Runtime > Change runtime type > T4 GPU** before running.

All logic lives in `src/slmbench/`; this notebook is operational glue.

In [ ]:
# 1. GPU check -- expect a Tesla T4
!nvidia-smi -L

In [ ]:
# 2. Secrets and repository
#    A public repository clones without credentials. GITHUB_TOKEN is
#    read from Colab secrets only if one is set, which is what a private
#    checkout needs; it is never printed either way.
from google.colab import userdata
import os, subprocess, sys

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN                 # picked up by huggingface_hub

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None

GH_OWNER = "abdullahajaz14"
GH_REPO  = "slm-quant-bench"       # the public code-only release
_auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
CLONE_URL = f"https://{_auth}github.com/{GH_OWNER}/{GH_REPO}.git"

# Always bring the checkout up to date. A restarted session can keep the
# runtime disk, so a checkout from an earlier attempt survives; cloning
# only when the directory is absent would then run new notebook cells
# against stale repository code, which is invisible and very confusing.
if os.path.isdir('/content/project'):
    subprocess.run(['git', '-C', '/content/project', 'remote', 'set-url',
                    'origin', CLONE_URL], check=True)
    subprocess.run(['git', '-C', '/content/project', 'fetch', '--quiet',
                    'origin', 'main'], check=True)
    subprocess.run(['git', '-C', '/content/project', 'reset', '--hard',
                    '--quiet', 'origin/main'], check=True)
    print("existing checkout updated to origin/main")
else:
    subprocess.run(['git', 'clone', '--quiet', CLONE_URL, '/content/project'],
                   check=True)
    print("repository cloned")

%cd /content/project/code/slm-quant-bench

# Print the commit actually in use, so the version running is never in
# doubt when reading logs after the fact.
head = subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                      capture_output=True, text=True).stdout.strip()
print("running commit:", head)

# Install the package itself. sys.path tricks are not enough: the run
# scripts are launched as separate processes, which do not inherit this
# notebook's sys.path, so slmbench must be importable from a clean
# interpreter.
!pip -q install -e .

check = subprocess.run(
    [sys.executable, "-c", "import slmbench; print(slmbench.__file__)"],
    capture_output=True, text=True)
if check.returncode != 0:
    raise SystemExit("slmbench is not importable from a subprocess, so the "
                     f"run scripts would fail:\n{check.stderr}")
print("slmbench importable from a subprocess:", check.stdout.strip())

!git config user.name  "Abdullah Abdullah"
!git config user.email "abdullahajaz14@users.noreply.github.com"
print("repository ready")

In [ ]:
# 3. Installs.
#    Building llama-cpp-python from source on a Colab CPU takes the best
#    part of an hour and the cost repeats every session, so install a
#    prebuilt CUDA wheel. Only some toolkit tags are published, and the
#    runtime's own version is often newer than any of them (Colab
#    currently reports 12.8, for which no wheel exists), so the
#    candidates below are the tags known to be published, newest first.
#    CUDA minor versions are driver-compatible, so a 12.5 build runs on
#    a 12.8 runtime.
#
#    --only-binary :all: is the important flag: without it pip silently
#    falls back to a source build, which looks like a slow install and
#    costs an hour before anyone notices.
import subprocess, sys

CANDIDATES = ["cu125", "cu124", "cu123", "cu122", "cu121"]

def gpu_ready() -> bool:
    probe = ("import llama_cpp, sys; "
             "sys.exit(0 if llama_cpp.llama_supports_gpu_offload() else 1)")
    return subprocess.run([sys.executable, "-c", probe],
                          capture_output=True).returncode == 0

installed = False
for tag in CANDIDATES:
    print(f"trying prebuilt wheel: {tag}")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--only-binary", ":all:",
         "llama-cpp-python", "--extra-index-url",
         f"https://abetlen.github.io/llama-cpp-python/whl/{tag}"],
        capture_output=True, text=True)
    if r.returncode == 0 and gpu_ready():
        print(f"installed prebuilt wheel from {tag}")
        installed = True
        break
    print(f"  {tag} unavailable or lacks GPU offload")

if not installed:
    print("no prebuilt wheel worked; building from source (this is slow)")
    !CMAKE_ARGS="-DGGML_CUDA=on" pip -q install --force-reinstall --no-cache-dir llama-cpp-python

import llama_cpp
print("llama-cpp-python", llama_cpp.__version__,
      "| GPU offload:", llama_cpp.llama_supports_gpu_offload())
assert llama_cpp.llama_supports_gpu_offload(), \
    "no GPU offload: the sweep would run on CPU and the timings would be meaningless"

!pip -q install datasets rouge-score psutil pyyaml "huggingface_hub[cli]" \
                torch transformers sentencepiece

In [ ]:
# 4. llama.cpp tools (GGUF converter + quantiser).
#
#    PINNED to the same commit used to build the artefacts measured on
#    the target device. Chapter 3's risk register commits the study to a
#    pinned llama.cpp version, and it matters here beyond tidiness: the
#    cross-backend agreement check compares device output against Colab
#    output, so both sides must quantise with identical tooling or the
#    check would be confounded by the toolchain itself.
#
#    Built with CUDA off and with bounded parallelism: the quantiser is
#    a CPU tool that needs no GPU support, and an unbounded -j exhausts
#    Colab's RAM and kills the session.
import os

LLAMACPP_COMMIT = "571d0d540df04f25298d0e159e520d9fc62ed121"   # 18 Jul 2026

if not os.path.isdir("/content/llama.cpp"):
    !git clone --quiet https://github.com/ggerganov/llama.cpp /content/llama.cpp
    !cd /content/llama.cpp && git checkout --quiet $LLAMACPP_COMMIT
    !cmake -S /content/llama.cpp -B /content/llama.cpp/build \
        -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=OFF \
        -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF \
        -DLLAMA_BUILD_SERVER=OFF > /dev/null
    !cmake --build /content/llama.cpp/build -j 2 --target llama-quantize > /dev/null

os.environ["LLAMACPP_DIR"] = "/content/llama.cpp"
!cd /content/llama.cpp && echo "llama.cpp at $(git rev-parse --short HEAD)"
!ls -la /content/llama.cpp/build/bin/llama-quantize

In [ ]:
# 5. PARAMETER CELL -- the only cell to change between sessions
MODEL = "phi3-mini"   # phi3-mini | gemma2-2b | llama32-3b | mistral7b

# "pilot" runs 25 items per task across all six tasks, which is the
# protocol gate before any full run; "full" runs the complete sweep.
MODE  = "pilot"       # pilot | full

In [ ]:
# 6. Download, convert to FP16 GGUF, quantise to Q8_0 and Q4_K_M.
#    Skipped automatically if this session already built them.
import os, glob

os.makedirs("models", exist_ok=True)
os.makedirs("weights", exist_ok=True)

if not os.path.exists(f"models/{MODEL}-q4_k_m.gguf"):
    !bash scripts/download_convert.sh $MODEL
else:
    print(f"{MODEL} GGUFs already present in this session")

built = sorted(glob.glob(f"models/{MODEL}-*.gguf"))
for f in built:
    print(f"  {f}  {os.path.getsize(f) / 1024**3:.2f} GB")

# Fail loudly rather than letting the run proceed and record nothing.
expected = {f"models/{MODEL}-{p}.gguf" for p in ("fp16", "q8_0", "q4_k_m")}
missing = sorted(expected - set(built))
if missing:
    raise SystemExit(f"model artefacts missing, cannot run: {missing}")
print(f"all three {MODEL} artefacts present")

In [ ]:
# 7. Pull any results produced by earlier sessions, so this run resumes
#    rather than repeating finished work.
!git pull --quiet --rebase || echo "pull skipped"
!ls -lh results/*.jsonl 2>/dev/null || echo "no results yet (first session)"

In [ ]:
# 8. The run, one precision at a time, pushing results after each.
#
#    Free-tier sessions are time-limited. The runner is resumable, but
#    only from what has actually been pushed, so a session that dies
#    mid-run would otherwise lose everything it had computed.
#
#    Every run's output is also captured to results/logs/ and pushed, so
#    a failure can be diagnosed from the repository without anyone
#    having to transcribe a screenshot.
import os, subprocess, datetime

CONFIG = ("configs/runs/pilot_colab.yaml" if MODE == "pilot"
          else "configs/runs/quality_colab.yaml")
os.makedirs("results/logs", exist_ok=True)
STAMP = datetime.datetime.now().strftime("%Y%m%d-%H%M")

def checkpoint(label):
    """Commit and push. Reports what actually happened: git push exits 0
    when there is nothing to send, so reporting on its exit code alone
    would claim success for a run that produced nothing, which is
    exactly how several empty runs went unnoticed."""
    subprocess.run(["git", "add", "results/"], check=False)
    staged = subprocess.run(["git", "diff", "--cached", "--name-only"],
                            capture_output=True, text=True).stdout.split()
    if not staged:
        print(f"  checkpoint {label}: NOTHING TO COMMIT "
              f"(the run produced no files)", flush=True)
        return
    subprocess.run(["git", "commit", "-q", "-m",
                    f"Colab {MODE} run: {MODEL} {label}"], check=False)
    subprocess.run(["git", "pull", "--quiet", "--rebase"], check=False)
    push = subprocess.run(["git", "push", "--quiet"],
                          capture_output=True, text=True)
    if push.returncode == 0:
        print(f"  checkpoint {label}: pushed {len(staged)} file(s)", flush=True)
    else:
        print(f"  checkpoint {label}: PUSH FAILED: "
              f"{push.stderr.strip()[:200]}", flush=True)

for precision in ["q4_k_m", "q8_0", "fp16"]:          # cheapest first
    log = f"results/logs/{MODEL}-{precision}-{STAMP}.log"
    print(f"\n=== {MODEL} {precision} -> {log} ===", flush=True)
    cmd = (f"python scripts/run_quality.py --config {CONFIG} "
           f"--models {MODEL} --precisions {precision}")
    rc = subprocess.run(f"{cmd} 2>&1 | tee {log}", shell=True).returncode

    # Keep the log small enough to be worth committing, but always keep
    # the tail, which is where a traceback lands.
    with open(log) as f:
        lines = f.readlines()
    if len(lines) > 400:
        with open(log, "w") as f:
            f.writelines(lines[:100] + ["\n... trimmed ...\n\n"] + lines[-300:])

    print(f"  exit code: {rc}", flush=True)
    checkpoint(precision)
    if rc != 0:
        print(f"  {precision} FAILED; log pushed for diagnosis", flush=True)

In [ ]:
# 9. Curated UK public-sector corpus. In pilot mode this is already
#    covered by pilot_colab.yaml, so it is skipped.
if MODE == "full":
    !python scripts/run_quality.py --config configs/runs/ukps_colab.yaml --models $MODEL
else:
    print("pilot mode: curated corpus already included in the pilot run")

In [ ]:
# The commit identity uses the GitHub noreply address, so no personal
# email address is published in this repository.
# 10. Final hand-off: confirm what is on the remote.
!git add results/ && git -c user.name="Abdullah Abdullah" \
   -c user.email="abdullahajaz14@users.noreply.github.com" \
   commit -q -m "Colab $MODE run: $MODEL (final)" || echo "already committed"
!git pull --quiet --rebase && git push --quiet && echo "all results pushed"
!wc -l results/*.jsonl

## After each session

`Runtime > Disconnect and delete runtime`, change `MODEL` in cell 5, and
run again. Results accumulate in the repo, so a dropped session is never
lost work.

## After the final session

Nothing to merge by hand: all four models' records are in
`results/quality-colab.jsonl` and `results/ukps-colab.jsonl` in the
repository. Locally, `git pull` then continue with the analysis,
the on-device efficiency battery and the agreement check.